Split by “speaker‐turn”

In [15]:
# Pseudocode: you can adapt to your file format
raw = open("/Users/saumyapandey/Downloads/CSTC_Interview_1_Participant_ID_72.txt").read().split("\n\n")
docs = []
for block in raw:
    # skip questions
    if block.startswith("Q"):
        continue
    # collect interviewer tags if you like, or drop them
    if block.startswith("Interviewer:"):
        continue
    docs.append(block.strip())


Clean up filler token

In [16]:
import re
docs = [re.sub(r"\b(um|uh+)\b", "", d, flags=re.IGNORECASE).strip()
        for d in docs]


Sentiment “Classification” via Prototype Similarity.

We’ll use ClinicalBERT purely for embeddings, then assign each segment to the nearest of three “prototype” vectors.


1. Load and Embed:

In [17]:
from transformers import AutoTokenizer, AutoModel
import torch
from sklearn.metrics.pairwise import cosine_similarity

model_name = "emilyalsentzer/Bio_ClinicalBERT"
tokenizer  = AutoTokenizer.from_pretrained(model_name)
model      = AutoModel.from_pretrained(model_name).eval()

def embed(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
    with torch.no_grad():
        last_hidden = model(**inputs).last_hidden_state
    return last_hidden[:,0].cpu().numpy()[0]  # [CLS] vector


2. Create your sentiment prototypes

In [18]:
prototypes = {
  "positive": embed("I’m very pleased with how it’s going."),
  "negative": embed("I’m frustrated and unhappy with this."),
  "neutral":  embed("It’s neither good nor bad; it’s just okay.")
}


3. Classify each segment

In [19]:
def classify_sentiment(seg):
    v = embed(seg)
    sims = {lbl: cosine_similarity([v],[p])[0,0]
            for lbl,p in prototypes.items()}
    return max(sims, key=sims.get)

sentiments = [classify_sentiment(d) for d in docs]

for seg, sent in zip(docs, sentiments):
    print(f"{sent.upper():>8} → {seg[:60]}...")


NEGATIVE → ﻿Reflective Interview
Participant ID #72
Time Point: 1...
 NEUTRAL → Q2: Great, wonderful, what are the main activities for you'r...
NEGATIVE → Annotations
1 The themes here can be matched, but they are n...


Thematic Extraction with BERTopic

BERTopic will cluster semantically similar segments (thanks to ClinicalBERT’s embeddings) and surface the top words per theme.

In [20]:
!pip install bertopic

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [21]:
!pip install --upgrade pip

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [22]:
import inspect
from collections import namedtuple

# If ArgSpec doesn’t exist (Python 3.12+), create a minimal stand-in
if not hasattr(inspect, "ArgSpec"):
    inspect.ArgSpec = namedtuple("ArgSpec", ["args", "varargs", "keywords", "defaults"])


In [26]:
from bertopic import BERTopic
from sklearn.preprocessing import normalize
import numpy as np


Build embedding matrix

In [27]:
embs = np.stack([embed(d) for d in docs])
embs = normalize(embs)  # often helps HDBSCAN


Fit the model

Inspect the themes

In [28]:

from umap import UMAP
from hdbscan import HDBSCAN

# 1) Simplify the embedding space:
umap_model = UMAP(
    n_neighbors=5,      # fewer neighbors → tighter local structure
    n_components=2,     # 2D is easier to cluster
    min_dist=0.1,       # allow some overlap between points
    metric="cosine"
)

# 2) Make HDBSCAN willing to accept small clusters:
hdbscan_model = HDBSCAN(
    min_cluster_size=4,      # allow clusters as small as 2 docs
    min_samples=1,           # fewer samples to form a cluster
    metric="euclidean",
    cluster_selection_method="eom"
)

topic_model = BERTopic(
    embedding_model=None, 
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    nr_topics=None,         # skip auto-reduce; just take raw clusters
    verbose=True
)

topics, probs = topic_model.fit_transform(docs, embs)

# Now inspect how many docs per cluster:
from collections import Counter
print(Counter(topics))


2025-06-24 05:16:21,060 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
/Users/saumyapandey/miniconda3/lib/python3.12/site-packages/umap/spectral.py:332: RuntimeWarning:

k >= N for N * N square matrix. Attempting to use scipy.linalg.eigh instead.

/Users/saumyapandey/miniconda3/lib/python3.12/site-packages/umap/spectral.py:332: RuntimeWarning:

k >= N for N * N square matrix. Attempting to use scipy.linalg.eigh instead.



TypeError: Cannot use scipy.linalg.eigh for sparse A with k >= N. Use scipy.linalg.eigh(A.toarray()) or reduce k.

In [ ]:
info = topic_model.get_topic_info()
print(info)


   Topic  Count                 Name  \
0      0     10  0_and_we_that_think   
1      1      4   1_like_the_you_and   

                                      Representation  \
0  [and, we, that, think, to, of, the, like, real...   
1  [like, the, you, and, that, to, its, know, thi...   

                                 Representative_Docs  
0  [I mean, I think I just outlined a bunch, but ...  
1  [Yes, I have encountered challenges it's a bro...  


getting top words per topic

In [ ]:
for topic_id in [0, 1]:
    words = topic_model.get_topic(topic_id)
    print(f"\nTopic {topic_id} keywords:")
    print([w for w, score in words][:10])



Topic 0 keywords:
['and', 'we', 'that', 'think', 'to', 'of', 'the', 'like', 'really', 'just']

Topic 1 keywords:
['like', 'the', 'you', 'and', 'that', 'to', 'its', 'know', 'think', 'of']


In [ ]:
import numpy as np

docs_arr = np.array(docs)
for topic_id in [0, 1]:
    print(f"\n--- Samples for Topic {topic_id} ---")
    sample_idxs = np.where(np.array(topics) == topic_id)[0][:3]
    for i in sample_idxs:
        print("•", docs_arr[i][:100].replace("\n", " "), "…")



--- Samples for Topic 0 ---
• A lot of it is project management getting people together. To talk about different procedural things …
• You mean my definition of it? …
• I think it means a lot of different things. Sometimes it's helping people understand, I’m sorry my m …

--- Samples for Topic 1 ---
• I think it's extremely complex, because of the nature of the group. So we are not, it’s a shared pow …
• Yes, I have encountered challenges it's a broad question.  , I think, like I said it's really just l …
• I think we've managed to do things that were vitally important to be done, we have kept the process  …


In [ ]:
import random
from collections import defaultdict

# 1. Sentiment sanity check: randomly sample ~10 segments
print("=== Sentiment Sanity Check ===\n")
n_samples = min(10, len(docs))
for idx in random.sample(range(len(docs)), n_samples):
    print(f"[{sentiments[idx].upper():7}] {docs[idx][:100]}…\n")

# 2. Theme coherence: for each discovered topic, show up to 5 examples
print("\n=== Theme Coherence Check ===\n")
# group indices by topic
by_topic = defaultdict(list)
for i, t in enumerate(topics):
    by_topic[t].append(i)

for topic_id, idxs in by_topic.items():
    if topic_id == -1:
        label = "Noise/Outliers"
    else:
        label = f"Topic {topic_id}"
    print(f"-- {label} ({len(idxs)} segments) --")
    # sample up to 5 examples for this topic
    for i in random.sample(idxs, min(5, len(idxs))):
        print(f" • {docs[i][:120]}…")
    print()


=== Sentiment Sanity Check ===

[POSITIVE] I just think it's good to be like him, I think, Robert Wood Johnson really helps me, and I think, ma…

[NEUTRAL] I mean, I think zoom makes everything hard. You can't sit in a zoom and have the same experience as …

[NEGATIVE] I think it means a lot of different things. Sometimes it's helping people understand, I’m sorry my m…

[POSITIVE] You mean my definition of it?…

[NEUTRAL] Yes, I have encountered challenges it's a broad question.  , I think, like I said it's really just l…

[POSITIVE] A lot of it is project management getting people together. To talk about different procedural things…

[NEGATIVE] I think it's extremely complex, because of the nature of the group. So we are not, it’s a shared pow…

[NEUTRAL] I mean you know, last year I’ve been here, yes, like spring as well.…

[NEUTRAL] I think we've managed to do things that were vitally important to be done, we have kept the process …

[NEUTRAL] I mean, I don't think we're, the implem

In [3]:
# -*- coding: utf-8 -*-
"""
Sentiment Analysis & Thematic Clustering
on a single “Reflective Interview” transcript (Participant #72),
printing every segment per topic.
"""

# 0) Monkey-patch to avoid UMAP → TensorFlow recursion errors
import sys, types
sys.modules['umap.parametric_umap'] = types.ModuleType('umap.parametric_umap')

# 1) Patch inspect.ArgSpec for Python 3.12+
import inspect
from collections import namedtuple
if not hasattr(inspect, "ArgSpec"):
    inspect.ArgSpec = namedtuple("ArgSpec", ["args","varargs","keywords","defaults"])

# 2) Imports & dependencies (install once):
#    pip install transformers torch scikit-learn bertopic umap-learn hdbscan
import re
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
from collections import Counter, defaultdict

# 3) Raw interview text
interview_text = """
Question 3:  opening right now, but prior to this in this whole process of getting this up and running, how do you think the problem management has been going?


P 39:so for us? The the main activities have been very low because of our client base. So you know, I I know that we do have somebody that participates in meetings at the City Mission over the project. But as far as how many clients are impacted we have a very low number of clients that would. you know, fit into the into the project mainly because they can't be followed past discharge due to, you know,  cfr hipaa regulations for chemical abuse clients so so they can't be followed in.you know. So once they're gone from us, we don't follow them at all.

P 60: Well, the part that I've been included on is really looking at healthy link alerts, how they can inform providers about transition and then those alerts informing both the transitional case managers at Jericho Road and the Health Home Care Coordinator, coordinator team about those transitions and then trying to work together to provide the best care for somebody transitioning out of the hospital and then making sure they have good care in an outpatient setting, moving forward. So how is it going? I think right now we've kind of been at the planning and discussion stage, which there's been a lot of great discussions and that we're kind of hopeful to start getting some alerts and working collaboratively together. You know, moving forward.

P 88:Well, I think that, I feel like, there's been some really positive thing. I think the development of the clinical team and of the Advisory Board has actually been going very well. That the people who have participated are interested in trying to do something different. I don't think they realize how different it's going to be.Um, but certainly they're trying very hard to adapt the kinds of things that they have done in the past to make them work here. Probably one of the people who it's easy for is someone who doesn't have any previous pattern.So, for example, there's a CLINICAN. That was never in Buffalo city mission prior to this. And so, that person is actually extremely flexible and willing to try a lot of different things, as opposed to people who are here for a long time and have a pattern of how they do things that needs to get changed. So it's been interesting to see some of the push and pull that occurs with that. Um, so I think that all of those are very positive and I'm really pleased at where we are right now.I think the area that still in most uncertain is figuring out how to get reimbursed for the respite service itself. So certainly, the physician services can get really reimbursed, the behavioral health people services can get reimbursed but how do we actually pay for the bet itself and the fact that they're supervision 24/7 for those patients. It's a little bit than it is for other people in the city mission. That's been an ongoing problem. It sounds like it's a problem in every one of these units based on what we're finding out, talking to different facilities. So I think that it will be, it will be an ongoing challenge. And I think that the city mission has actually bent over backwards to try and make this work and are planning to put together a pilot where there's a much-reduced rate, initially, so that we can develop some data that will allow us to demonstrate that the project is being effective.

P 94. Well, no, I think that work group is good, it's, it's really good you know because basically we look at it from the people that are going to be working the medical respite you know, trying to think out the pros and the cons. What could go wrong. We are trying to assist with that and to make it as smooth and collaborative relationship with HEALTH SYSTEM and the city mission. I think there is a disconnect sometimes with the Advisory Board and who works with the city mission and the hospital and the contract and us. I think that, you know, there was a lot of negotiations going on and HEALTH SYSTEM hadn't signed a contract and we were doing education to a group, like well as far as we know, we're not participating in this. Where I knew, the bigwig said, we were going to, you know what I mean, and I’m telling him we're going to participate and we're going to have to do it this way. I'm like how come this isn't a little more transparent. You know what I mean, because you know that I think the tipping scale was the hospital, the two emergency rooms I work in one there's the other one, a couple of weeks ago or her holding almost 50 people to be admitted and there might have been the tipping point that we have to find a way to get patients that are just holding here for placement kind of issues, out. And that might have been the tipping point and I just think that sort of, It should have been a little more transparent with the work group and like the status of it. 

P 69. I think the current, the current main activities are getting the logistics figured out for the pieces of data that we can provide that are wanted to be provided, and who's going to provide them, and how we're going to put it all together and deliver it. I think that I think it's been a good conversation, as far as the research where you know there's always the research, I'm not going to call it a bubble, but it's an ideal where it meets the realities of capability and then I think we're. But I think we're on a on our way to a good product for the research project and for alerts in general. So, I think it's beneficial to both of us, too, I think, you know, I think we're going to get some good data in some good and some good products for you know, for Sharon and her team plus, plus, you know, this is something that we're thinking of advancing as well as more advanced notifications like this. So I think it's been so far, so so good about it. I think that's where we are right now. Is the logistics of the of the product itself.

P 26: So main activities would include: Some reports being generated by Jericho road with Sharon's help, so those are what we are referring to as a high needs report. Jericho Road also gives us what we refer to as a subscribe and notify list. Then on the HEALTHeLINK side, those reports need to be ingested into a database table where some, and I’m not sure if the algorithm portion of this magic is happening on Sharon side or the healthElink side, but there's an algorithm being built to kind of put some data together on patients that would be useful for these enhanced care alerts and then, once the database is populated with these reports, HEL will build a channel which is just an interface engine that queries the database, and pulls these different fields out from the different tables, and puts it together in a nice, pretty direct mail message, and we send that off to Jericho. So the triggering mechanism of that is an ADT message for one of Jericho Road's patients. So, if one of Jericho road's, patients was seen at a hospital, HEL gets that ADT, we go ahead and query these tables that have been populated with these reports, and then we spit out this enhanced care alert and send it via direct mail. So how do I think it's going? It's going. Okay. It was. It was a rough start, but I think we're in a good in a good spot now. I think it was a little chaotic just because of maybe some of the project management, or some disconnects on our side, or what teams were responsible for what? But we kind of had a regrouping of that, and brought it all back together, and now I think we have a clear path forward. So I’m feeling pretty good about it now.

P 91. Yeah, this enhanced care alert, I guess, is the the main one, and I believe it's about to be released into the wild here shortly. How’s it going? I think, from my perspective it's going well, it's going fine, and then lots of complicated moving pieces. But I think we're making good progress.

P 84: Well, the thing is Jericho Road is so understaffed that I mean she could be doing it almost, I don't know half time, probably 20 to 30 hours a week and Worker(?)could stay caught up but, they only allow her to do it like a couple of days a week, because she works in other areas. So, we never get caught up. And I only do 12 hours for Jericho Road.Well, I mean it’s an ongoing process. There's always patients being discharged. We just start 3 days out, every time we start on the list, so the ones that go by that we haven't looked at for Wednesday, Thursday, or Friday, we just let them go and then we start again. Like every 3 days, or I'll take it out to 4 or 5. I took it out to 4 or 5 today.

P 99: Sure, the current main activities that I'm involved in is just working on building an expanded high needs report based off of our initial criteria. That's some additional criteria that Dr.ABC gave us. So we've been kind of rolling it out, and we finally have like a good final project. So it's nice to like to see the project come to fruition, especially since it's been well over a year. But Dr.ABC has been like absolutely great to work with. She's like super kind and pleasant. At times I feel like she wants to be like okay you like you dumbass that's not what I asked for, and you gave me this and this is the product that you gave me. But she, like is always says in the kind of way to saying like, hey we might need like additional iteration, and things like that. But it's just nice to see that the projects like we’re like almost to like the implementation of it. But it's been a long project, I think.

P 72:  Um you know it's interesting I, I guess I, I think I get the sense that there's been, I don't know that I fully understand the whole process. So, but I think what is happening is that there is like a working group that's deciding on things, and then that is some of those things, has been have been presented to the advisory committee and then like I’ve made some changes that related to our organization based on, based on being able to like review it a second time I guess before it was like, finally approved. So I mean, I think that, yeah it's interesting, there's I think there's multiple things related to the to the RCU and sometimes they've gotten like, in my head, they've gotten mixed up. Like this particular project being led by a CLINCIAL SCHLOAR is different than like some of the other things with the city mission.1

P 79: ​​Well. I it sounds like we're in the phase where we're looking to kind of deploy the care alert based off yesterday's meeting. It seems like there the wheels are falling off a little bit. Yeah, I think we have some interesting impressions just based off of yesterday's meeting and kind of like the appropriateness of each program involved. I mean so for the behavioral health provider it kind of sounds like they might not have a lot of clients in the population and then it sounds like Jericho Road isn't really ready. So yeah, that's kind of our understanding is, we're just we feel like, just in a holding pattern and going to see kind of what happens.


"""

# 4) Extract only participant responses into `docs`
docs = []
for line in interview_text.splitlines():
    line = line.strip()
    if not line:
        continue
    if line.startswith("Q") or line.startswith("Interviewer"):
        continue
    if line.startswith("Participant 79") or line.startswith("participant 79") or line.startswith("Time Point"):
        continue
    docs.append(line)

# 5) Clean out filler tokens
docs = [re.sub(r"\b(um|uh+)\b", "", seg, flags=re.IGNORECASE).strip() for seg in docs]

# 6) Load ClinicalBERT & define embed()
tokenizer = AutoTokenizer.from_pretrained("emilyalsentzer/Bio_ClinicalBERT")
model     = AutoModel.from_pretrained("emilyalsentzer/Bio_ClinicalBERT").eval()

def embed(text: str) -> np.ndarray:
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
    with torch.no_grad():
        last_hidden = model(**inputs).last_hidden_state
    return last_hidden[:,0].cpu().numpy()[0]

# 7) Build sentiment prototypes & classify
prototypes = {
    "positive": embed("I’m very pleased with how it’s going."),
    "negative": embed("I’m frustrated and unhappy with this."),
    "neutral":  embed("It’s neither good nor bad; it’s just okay.")
}

def classify_sentiment(seg: str) -> str:
    v    = embed(seg)
    sims = {lbl: cosine_similarity([v],[p])[0,0] for lbl,p in prototypes.items()}
    return max(sims, key=sims.get)

sentiments = [classify_sentiment(d) for d in docs]

print("\n--- Sentiment Analysis ---")
for seg, s in zip(docs, sentiments):
    print(f"[{s.upper():7}] {seg}")

# 8) Build & normalize embeddings
embs = np.stack([embed(d) for d in docs])
embs = normalize(embs)

# 9) Configure UMAP (random init) & HDBSCAN for BERTopic
umap_model = UMAP(n_neighbors=5, n_components=2, min_dist=0.1,
                  metric="cosine", init="random")
hdbscan_model = HDBSCAN(min_cluster_size=2, min_samples=1,
                        metric="euclidean", cluster_selection_method="eom")

topic_model = BERTopic(
    embedding_model=None,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    nr_topics=None,
    verbose=False
)

# 10) Fit & then PRINT ALL segments per topic
topics, probs = topic_model.fit_transform(docs, embs)

print("\n--- Document Count per Topic ---")
print(Counter(topics))

info = topic_model.get_topic_info()
print("\n--- Topic Info ---")
print(info)

print("\n--- Top Keywords per Topic ---")
for tid in info.Topic.unique():
    if tid == -1: continue
    words = topic_model.get_topic(tid)
    print(f"Topic {tid}:", [w for w,_ in words][:10])

print("\n--- All Segments per Topic ---")
docs_arr = np.array(docs)
by_topic = defaultdict(list)
for i, t in enumerate(topics):
    by_topic[t].append(i)

for tid, idxs in by_topic.items():
    label = "Noise" if tid == -1 else f"Topic {tid}"
    print(f"\n-- {label} ({len(idxs)} segments) --")
    for i in idxs:
        print(" •", docs_arr[i])
        
with open("topic_analysisPID69_HeatheLink.txt","w", encoding="utf-8") as f:
    for tid, idxs in by_topic.items():
        label = "Noise" if tid == -1 else f"Topic {tid}"
        f.write(f"\n-- {label} ({len(idxs)} segments) --\n")
        for i in idxs:
            f.write(f" • {docs_arr[i]}\n")
print("Wrote full analysis to topic_analysis.txt")



--- Sentiment Analysis ---
[NEUTRAL] P 39:so for us? The the main activities have been very low because of our client base. So you know, I I know that we do have somebody that participates in meetings at the City Mission over the project. But as far as how many clients are impacted we have a very low number of clients that would. you know, fit into the into the project mainly because they can't be followed past discharge due to, you know,  cfr hipaa regulations for chemical abuse clients so so they can't be followed in.you know. So once they're gone from us, we don't follow them at all.
[NEUTRAL] P 60: Well, the part that I've been included on is really looking at healthy link alerts, how they can inform providers about transition and then those alerts informing both the transitional case managers at Jericho Road and the Health Home Care Coordinator, coordinator team about those transitions and then trying to work together to provide the best care for somebody transitioning out of the

In [2]:
# -*- coding: utf-8 -*-
"""
Sentiment Analysis & Thematic Clustering
on a single “Reflective Interview” transcript (Participant #72),
printing every segment per topic.
"""

# 0) Monkey-patch to avoid UMAP → TensorFlow recursion errors
import sys, types
sys.modules['umap.parametric_umap'] = types.ModuleType('umap.parametric_umap')

# 1) Patch inspect.ArgSpec for Python 3.12+
import inspect
from collections import namedtuple
if not hasattr(inspect, "ArgSpec"):
    inspect.ArgSpec = namedtuple("ArgSpec", ["args","varargs","keywords","defaults"])

# 2) Imports & dependencies (install once):
#    pip install transformers torch scikit-learn bertopic umap-learn hdbscan
import re
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
from collections import Counter, defaultdict

# 3) Raw interview text
interview_text = """
P 39:Main activities very low due client base. Somebody participates meetings City Mission over project. Few clients impacted, mainly because can't followed past discharge due CFR HIPAA regulations chemical abuse clients. Once gone, not followed.
P 60:Included looking Healthy Link alerts, informing providers about transition, alerts informing transitional case managers Jericho Road, Health Home Care Coordinator team about transitions, trying work together provide best care somebody transitioning hospital, ensuring good care outpatient setting moving forward. Going: currently planning, discussion stage. Great discussions. Hopeful start getting alerts, working collaboratively moving forward.
P 88:Positive things. Development clinical team, Advisory Board going well. Participants interested doing something different. Don’t realize how different. Trying hard adapt past practices work here. Easier someone without previous pattern. Clinician never Buffalo City Mission prior, extremely flexible, willing try different things. Others long-time patterns harder change. Push, pull dynamics. Pleased where now. Uncertainty reimbursement respite service. Physician services reimbursed, behavioral health reimbursed — how pay bed, 24/7 supervision? Different city mission others. Ongoing problem, common all units. City Mission trying make work, planning pilot reduced rate, develop data demonstrate project effectiveness.
P 94:Work group good. View medical respite, consider pros, cons, prevent issues. Aim smooth, collaborative relationship HEALTH SYSTEM, City Mission. Disconnect Advisory Board, City Mission, hospital, contract, us. Negotiations ongoing. HEALTH SYSTEM hadn’t signed contract. Educating group unsure participation. Leadership agreed participation. Transparency lacking. Tipping point: emergency rooms holding 50 patients, placement issues. Prompted urgency. Transparency needed work group, status.
P 69:Current activities: logistics, identifying data pieces, providers, integration, delivery. Research conversations — ideal vs. reality capability. On way good product research, alerts. Beneficial both sides. Expecting good data, good products Sharon, team. Advancing notifications. So far, so good. Currently: product logistics.
P 26:Main activities: reports generated Jericho Road, Sharon's help — high needs report. Subscribe, notify list. HEALTHeLINK side: ingest reports database, algorithm (Sharon’s/HEL’s side) assembles useful patient data enhanced care alerts. Database populated → HEL builds channel (interface engine) → queries database → pulls fields → creates alert → sends direct mail Jericho. Trigger: ADT message Jericho Road patient. If seen hospital → HEL receives ADT → queries tables → sends enhanced alert.Going okay. Rough start. Chaotic project management, unclear responsibilities. Regrouped. Clear path forward. Feeling good now.
P 91:Enhanced care alert main focus. About released. Going: well, fine. Complicated moving pieces. Good progress.
P 84:Jericho Road understaffed. Worker could catch up working half time (20–30 hours/week). Allowed couple days/week only. Never caught up. Personal input: 12 hours/week. Ongoing process. Patients constantly discharged. Restart list every 3 days. Missed ones skipped. Sometimes 4–5 days out.
P 99:Current activity: building expanded high needs report based initial, additional criteria (Dr. ABC). Rolling out. Final project ready. Project coming together after year+. Dr. ABC great, kind. Even mistakes addressed kindly — suggests iterations nicely. Nice seeing implementation near.
P 72:Interesting. Don’t fully understand process. Working group deciding things → presented advisory committee. Made changes organization based second review. Project led CLINICAL SCHOLAR different other City Mission projects. Things mix up sometimes.
P 79:Phase: deploying care alert. Based yesterday’s meeting — wheels falling off. Impressions: behavioral health provider has few relevant clients, Jericho Road not ready. Holding pattern. Observing developments.

"""

# 4) Extract only participant responses into `docs`
docs = []
for line in interview_text.splitlines():
    line = line.strip()
    if not line:
        continue
    if line.startswith("Q") or line.startswith("Interviewer"):
        continue
    if line.startswith("Participant 79") or line.startswith("participant 79") or line.startswith("Time Point"):
        continue
    docs.append(line)

# 5) Clean out filler tokens
docs = [re.sub(r"\b(um|uh+)\b", "", seg, flags=re.IGNORECASE).strip() for seg in docs]

# 6) Load ClinicalBERT & define embed()
tokenizer = AutoTokenizer.from_pretrained("emilyalsentzer/Bio_ClinicalBERT")
model     = AutoModel.from_pretrained("emilyalsentzer/Bio_ClinicalBERT").eval()

def embed(text: str) -> np.ndarray:
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
    with torch.no_grad():
        last_hidden = model(**inputs).last_hidden_state
    return last_hidden[:,0].cpu().numpy()[0]

# 7) Build sentiment prototypes & classify
prototypes = {
    "positive": embed("I’m very pleased with how it’s going."),
    "negative": embed("I’m frustrated and unhappy with this."),
    "neutral":  embed("It’s neither good nor bad; it’s just okay.")
}

def classify_sentiment(seg: str) -> str:
    v    = embed(seg)
    sims = {lbl: cosine_similarity([v],[p])[0,0] for lbl,p in prototypes.items()}
    return max(sims, key=sims.get)

sentiments = [classify_sentiment(d) for d in docs]

print("\n--- Sentiment Analysis ---")
for seg, s in zip(docs, sentiments):
    print(f"[{s.upper():7}] {seg}")

# 8) Build & normalize embeddings
embs = np.stack([embed(d) for d in docs])
embs = normalize(embs)

# 9) Configure UMAP (random init) & HDBSCAN for BERTopic
umap_model = UMAP(n_neighbors=5, n_components=2, min_dist=0.1,
                  metric="cosine", init="random")
hdbscan_model = HDBSCAN(min_cluster_size=2, min_samples=1,
                        metric="euclidean", cluster_selection_method="eom")

topic_model = BERTopic(
    embedding_model=None,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    nr_topics=None,
    verbose=False
)

# 10) Fit & then PRINT ALL segments per topic
topics, probs = topic_model.fit_transform(docs, embs)

print("\n--- Document Count per Topic ---")
print(Counter(topics))

info = topic_model.get_topic_info()
print("\n--- Topic Info ---")
print(info)

print("\n--- Top Keywords per Topic ---")
for tid in info.Topic.unique():
    if tid == -1: continue
    words = topic_model.get_topic(tid)
    print(f"Topic {tid}:", [w for w,_ in words][:10])

print("\n--- All Segments per Topic ---")
docs_arr = np.array(docs)
by_topic = defaultdict(list)
for i, t in enumerate(topics):
    by_topic[t].append(i)

for tid, idxs in by_topic.items():
    label = "Noise" if tid == -1 else f"Topic {tid}"
    print(f"\n-- {label} ({len(idxs)} segments) --")
    for i in idxs:
        print(" •", docs_arr[i])
        
with open("topic_analysisPID69_HeatheLink.txt","w", encoding="utf-8") as f:
    for tid, idxs in by_topic.items():
        label = "Noise" if tid == -1 else f"Topic {tid}"
        f.write(f"\n-- {label} ({len(idxs)} segments) --\n")
        for i in idxs:
            f.write(f" • {docs_arr[i]}\n")
print("Wrote full analysis to topic_analysis.txt")



--- Sentiment Analysis ---
[POSITIVE] P 39:Main activities very low due client base. Somebody participates meetings City Mission over project. Few clients impacted, mainly because can't followed past discharge due CFR HIPAA regulations chemical abuse clients. Once gone, not followed.
[NEGATIVE] P 60:Included looking Healthy Link alerts, informing providers about transition, alerts informing transitional case managers Jericho Road, Health Home Care Coordinator team about transitions, trying work together provide best care somebody transitioning hospital, ensuring good care outpatient setting moving forward. Going: currently planning, discussion stage. Great discussions. Hopeful start getting alerts, working collaboratively moving forward.
[NEUTRAL] P 88:Positive things. Development clinical team, Advisory Board going well. Participants interested doing something different. Don’t realize how different. Trying hard adapt past practices work here. Easier someone without previous pattern. 

In [ ]:
!pip install "protobuf<=3.20.3"


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [ ]:
!pip install --upgrade "keras>=3.5.0"



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 3.4 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: keras
    Found existing installation: keras 2.11.0
    Uninstalling keras-2.11.0:
      Successfully uninstalled keras-2.11.0


zsh:1: command not found: Y
